# Lecture 16

## Polars

## Week 6 Monday

## Miles Chen, PhD

[https://pola.rs/](https://pola.rs/)

# Why Polars?

Polars is built for speed.

Polars can use lazy execution. The optimizer looks at the entire query and builds an optimized plan for processing it.

## Stick to Pandas for:

- Small/interactive analysis: For quick data exploration and when high performance doesn't matter, pandas is perfectly fine.
- Plotting & convenience: Many plotting libs integrate directly with pandas; Polars usually hands off to pandas or converts to arrays for plotting.

## Quick Translation Guide
If you are coming from R/`dplyr` or SQL, Polars will feel very familiar once you map the verbs:

| Polars | R (`dplyr`) | SQL |
| :--- | :--- | :--- |
| `.select()` | `select()` | `SELECT` |
| `.filter()` | `filter()` | `WHERE` |
| `.with_columns()` | `mutate()` | `SELECT ... AS` |
| `.group_by().agg()` | `group_by() %>% summarize()` | `GROUP BY` |

In [1]:
import polars as pl
pl.__version__

RuntimeError: unknown feature flag: 'sse3'


## 1) Load the Titanic dataset

* survived: 0 = No, 1 = Yes
* pclass: Ticket class: 1 = 1st, 2 = 2nd, 3 = 3rd
* sibsp: number of siblings and/or spouses aboard
* parch: number of parents and/or children aboard
* ticket: ticket number
* cabin: cabin number
* embarked: port of embarkation: C = Cherbourg, Q = Queenstown, S = Southampton


In [ ]:
# Read in the data eagerly (materializes immediately)
titanic = pl.read_csv("../data/titanic.csv")

titanic

## 2) Eager vs. Lazy

- **Eager** executes now and returns results.
- **Lazy** builds a plan (`LazyFrame`) and executes at `.collect()`, enabling optimizations.

The recommended method is to use lazy API. Polars will look at the entire query and optimize it before running. The lazy API allows you to work with datasets that are larger than your computer's memory.

To use the lazy API, use `pl.scan_csv()` instead of `pl.read_csv()`

We'll come back to this topic later, *after* we have introduced operational features in Polars.



## 3) Select, Rename, and Method Chaining

### Method Chaining
Notice that Polars operations return a *new* DataFrame. This allows you to chain multiple methods together sequentially (similar to the `%>%` or `|>` pipe in R) without constantly overwriting variables.

### Select and Rename
Select columns (also called series) using `name_of_dataframe.select("column_name")`

Multiple columns can be selected by passing a list of column names.

Columns or series can be renamed using `pl.col("current_name").alias("new_name")`

If you ever want to use attributes or methods for a column, you must use `pl.col("column_name")` and not just the string name.


In [ ]:
titanic.select([
    "survived",
    pl.col("pclass").alias("passenger_class"),
    "sex",
    "age",
    pl.col("fare").alias("fare_gbp")
])

### Using Polars Selectors (`polars.selectors`)
For selecting multiple columns based on their data type or name string, the `polars.selectors` module is incredibly efficient (very similar to `tidyselect` helpers like `starts_with()` or `where(is.numeric)` in R).

By community convention, we import it as `cs`.

In [ ]:
import polars.selectors as cs

# Example: Select only the numeric columns
titanic.select(cs.numeric()).head(3)


## 4) Filter rows and Sort rows


In [ ]:
# Example: females in 1st/2nd class, sorted by fare descending
titanic.filter(
    (pl.col("sex") == "female") & (pl.col("pclass") <= 2)
).sort("fare", descending=True)


## 5) `with_columns()` to create new columns using Expressions

`dataframe_name.with_columns()` will add columns to the data frame.

Provide a list of newly defined columns using expressions.

Name the columns with `.alias()`

One of the example expressions below is a `when-then-otherwise`.
Learn more at: [Polars When-Then-Otherwise Docs](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.when.html)

> **NOTE on `pl.lit()`:** We use `.lit()` to return a literal value. If you forget this and just write `.then("adult")`, Polars will search your dataset for a *column* named "adult" and throw a "Column not found" error! Always wrap raw strings or numbers in `pl.lit()` when building expressions.

In [ ]:
titanic_augmented = titanic.with_columns([
    (pl.col("sibsp") + pl.col("parch")).alias("family_size"),
    pl.when(pl.col("age") >= 18)
      .then(pl.lit("adult"))
      .otherwise(pl.lit("child"))
      .alias("age_group")
])
titanic_augmented

### Window Expressions

`.over()` will compute an expression over given groups.

See the example below where mean is calculated for all passengers, and then calculated for each passenger class.

In [ ]:
titanic.with_columns(
    pl.col("fare").mean().alias("mean_fare"),
    pl.col("fare").mean().over("pclass").alias("mean_fare_by_pclass")
)


## 6) Handling Missing Data

Use `is_null`, `is_not_null` to identify missing values.

Use `fill_null` to fill missing values. This can be paired with `.over()` to fill with averages calculated for groups.


In [ ]:
# Identify missing ages
missing_age = titanic.select([
    "passengerid",
    "age",
    pl.col("age").is_null().alias("age_is_missing")
])
missing_age

In [ ]:
# Example: fill missing age with class/sex mean
filled_age = titanic.with_columns(
    pl.col("age").fill_null(
        pl.col("age").mean().over(["pclass","sex"])
    ).alias("age_filled")
)
filled_age


## 7) Grouped Aggregations

Use `.group_by()` to create groups. Then use `.agg()` to specify expressions for aggregate (summary) values.


In [ ]:
titanic.group_by(["pclass","sex"]).agg([
    pl.col("survived").mean().alias("survival_rate"),
    pl.col("fare").mean().alias("avg_fare"),
    pl.col("age").mean().alias("avg_age"),
    pl.len().alias("n")
]).sort(["pclass","sex"])


## 8) Joins

You can join your data with another table.

Here we map `embarked` codes to city names. 

*(Note: When performing a left or inner join, Polars automatically drops the duplicate key column from the right table, keeping your DataFrame clean).* 

https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.join.html

In [ ]:
ports = pl.DataFrame({
    "embarked": ["C","Q","S"],
    "port_name": ["Cherbourg","Queenstown","Southampton"]
})

joined = titanic.join(ports, on="embarked", how="left")
joined

## 9) Reshaping: Pivot and Unpivot

Compute a **survival rate** table by `sex and pclass`.

We then pivot it (equivalent to `pivot_wider`), then unpivot it back (`pivot_longer`).


In [ ]:
# First, we make a summary table of survival rate by sex and pclass
survival = (
    titanic.group_by(["sex","pclass"])
           .agg(pl.col("survived").mean().alias("survival_rate"))
).sort(['pclass','sex'])
survival

In [ ]:
# arguments:
# on is the column name that contains the values that will become column headings
survival_pivot = survival.pivot(
        on="pclass",
        index="sex",
        values="survival_rate"
        )
survival_pivot

In [ ]:
# unpivot is like pivot_longer
# on takes a list of column names that will be unpivoted
survival_long = survival_pivot.unpivot(on=['1','2','3'], index = "sex", variable_name="pclass", value_name="survival_rate")
survival_long


## 11) Lazy Optimizations (Predicate/Projection Pushdown)

Lazy execution allows Polars to map out the most efficient computational path before running the code.

Once the `LazyFrame` query is constructed, we trigger the calculation using `.collect()`.

**Handling Massive Data:** Standard `.collect()` optimizes the query plan but still attempts to pull the final result into memory all at once. If you are working with datasets truly larger than your computer's RAM, add the argument `streaming=True` (e.g., `lazy_query.collect(streaming=True)`). This allows Polars to process massive datasets in manageable chunks behind the scenes without crashing your environment.

In [ ]:
# Lazy scan of the CSV file
titanic_lazy = pl.scan_csv("../data/titanic.csv")

lazy_query = (
  titanic_lazy
  .select(["survived","pclass","sex","fare"])
  .filter((pl.col("fare") > 30) & (pl.col("pclass") <= 2))
)


In [ ]:
lazy_query # reveals Polars plan to run the query

In [ ]:
lazy_query.collect()

In [ ]:
lazy_query2 = (
    titanic_lazy
    .filter(pl.col("age").is_not_null())
    .with_columns(
        (pl.col("age") >= 18).alias("adult"),
    )
    .group_by(["pclass", "sex"])
    .agg(
        survival_rate = pl.col("survived").mean(),
        avg_age       = pl.col("age").mean(),
        avg_fare      = pl.col("fare").mean(),
        prop_adult    = pl.col("adult").mean(),
        n             = pl.len()
    )
    .sort(["pclass", "sex"])
)

In [ ]:
# Execute the larger plan. We can pass streaming=True here to process in batches if the data was massive.
# not necessary here but demonstrates the option for larger datasets.
lazy_query2.collect(streaming=True)